In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import os

# Tắt cảnh báo chia cho 0 do dữ liệu mất cân bằng
warnings.filterwarnings('ignore', category=UndefinedMetricWarning)

# ==========================================
# 1. ĐỌC DỮ LIỆU & KIỂM TRA PHÂN BỐ LỚP
# ==========================================
# Lưu ý: Thay đổi đường dẫn file nếu cần
df = pd.read_csv('/kaggle/input/datasets/yasserh/wine-quality-dataset/WineQT.csv')

print("📊 Phân bố điểm chất lượng gốc (quality):")
print(df['quality'].value_counts().sort_index())

# Chuyển đổi sang nhóm: 0: Kém (<5), 1: Trung bình (5-6), 2: Tốt (>=7)
def map_quality_group(q):
    if q <= 4: return 0
    elif q <= 6: return 1
    else: return 2

df['quality_group'] = df['quality'].apply(map_quality_group)

print("\n📊 Phân bố lớp sau khi gộp (0: Kém | 1: Trung bình | 2: Tốt):")
print(df['quality_group'].value_counts().sort_index())

# ==========================================
# 2. CHUẨN BỊ DỮ LIỆU & CHUẨN HÓA
# ==========================================
X = df.drop(columns=['Id', 'quality', 'quality_group'])
y = df['quality_group']

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Chuẩn hóa dữ liệu (Rất quan trọng cho KNN và Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Đã chuẩn hóa dữ liệu thành công.")

# ==========================================
# 3. TỐI ƯU HÓA KNN & HUẤN LUYỆN MÔ HÌNH
# ==========================================

# --- BƯỚC MỚI: Tìm k tốt nhất cho KNN ---
print("\n🔍 Đang tìm kiếm tham số k tối ưu cho KNN...")
best_k = 5
best_acc_knn = 0
for k in range(3, 15): # Thử k từ 3 đến 14
    knn_temp = KNeighborsClassifier(n_neighbors=k, weights='distance')
    knn_temp.fit(X_train_scaled, y_train)
    acc_temp = knn_temp.score(X_test_scaled, y_test)
    print(f"  k={k} | Accuracy: {acc_temp:.4f}")
    if acc_temp > best_acc_knn:
        best_acc_knn = acc_temp
        best_k = k

print(f"✅ Chọn k = {best_k} cho KNN (Accuracy: {best_acc_knn:.4f})\n")

# Định nghĩa các mô hình (Sử dụng best_k vừa tìm được)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    f"KNN (k={best_k})": KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='euclidean')
}

best_model = None
best_score = 0

for name, model in models.items():
    print(f"{'='*40}")
    print(f" HUẤN LUYỆN: {name}")
    print(f"{'='*40}")
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    
    # Evaluate (Thêm zero_division=0 để tránh lỗi khi 1 lớp không có mẫu dự đoán)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print(f"Accuracy : {acc:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Kém', 'Trung bình', 'Tốt'], zero_division=0))
    
    # Lưu mô hình tốt nhất
    if acc > best_score:
        best_score = acc
        best_model = model
        best_model_name = name

print(f"\n🏆 MÔ HÌNH TỐT NHẤT: {best_model_name} (Accuracy: {best_score:.4f})")

# ==========================================
# 4. LƯU MÔ HÌNH
# ==========================================
os.makedirs('/kaggle/working', exist_ok=True)

# Lưu model tốt nhất và scaler
joblib.dump(best_model, '/kaggle/working/wine_quality_best_model.pkl')
joblib.dump(scaler, '/kaggle/working/scaler.pkl')

print("\n✅ Đã lưu thành công!")
print(f"📁 Model: /kaggle/working/wine_quality_best_model.pkl")
print(f"📁 Scaler: /kaggle/working/scaler.pkl")

📊 Phân bố điểm chất lượng gốc (quality):
quality
3      6
4     33
5    483
6    462
7    143
8     16
Name: count, dtype: int64

📊 Phân bố lớp sau khi gộp (0: Kém | 1: Trung bình | 2: Tốt):
quality_group
0     39
1    945
2    159
Name: count, dtype: int64

✅ Đã chuẩn hóa dữ liệu thành công.

🔍 Đang tìm kiếm tham số k tối ưu cho KNN...
  k=3 | Accuracy: 0.8908
  k=4 | Accuracy: 0.8952
  k=5 | Accuracy: 0.8996
  k=6 | Accuracy: 0.8908
  k=7 | Accuracy: 0.9039
  k=8 | Accuracy: 0.8952
  k=9 | Accuracy: 0.8996
  k=10 | Accuracy: 0.8996
  k=11 | Accuracy: 0.8996
  k=12 | Accuracy: 0.9039
  k=13 | Accuracy: 0.9039
  k=14 | Accuracy: 0.9039
✅ Chọn k = 7 cho KNN (Accuracy: 0.9039)

 HUẤN LUYỆN: Logistic Regression
Accuracy : 0.5895
F1-Score : 0.6581

Classification Report:
              precision    recall  f1-score   support

         Kém       0.08      0.62      0.15         8
  Trung bình       0.94      0.55      0.69       189
         Tốt       0.45      0.81      0.58        32

    